# Лабораторная №8 — Сегментация HITL (U-Net)

Этапы методички: 1. загрузка данных с Google Drive; 2. визуализация; 3. предобработка и дополнение кода; 4. построение U-Net для спутниковых снимков; 5. обучение; 6. тестирование; 7. результаты. **Нужны реальные файлы из ссылки в PDF**. Если в данных нет папки test, используем разбиение train без повторного использования одних и тех же изображений для обучения и тестирования.

## 1. Подключение Google Drive и загрузка библиотек

In [ ]:
import numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
from PIL import Image
from google.colab import drive
tf.keras.utils.set_random_seed(42)
drive.mount("/content/drive")

### 1.1. Чтение пар изображение–маска из каталогов методички

Для каждой картинки ищем маску с таким же именем без расширения; несоответствия не игнорируются молча. Изображения уменьшаются до 128×128 для Colab; маски масштабируются nearest-neighbor.

In [ ]:
def load_pairs(root,split,size=(128,128)):
    root=Path(root)
    # The folder structure is prescribed by the manual.
    img_dir=root/split/("image" if root.name=="ISBI-2012" else "images")
    mask_dir=root/split/("label" if root.name=="ISBI-2012" else "masks")
    assert img_dir.exists() and mask_dir.exists(),f"Missing folders: {img_dir}, {mask_dir}"
    images={p.stem:p for p in img_dir.iterdir() if p.suffix.lower() in (".png",".jpg",".jpeg",".tif",".tiff")}
    masks={p.stem:p for p in mask_dir.iterdir() if p.suffix.lower() in (".png",".jpg",".jpeg",".tif",".tiff")}
    keys=sorted(set(images)&set(masks))
    assert keys,f"No matching image/mask stems in {img_dir} and {mask_dir}"
    xs,ys=[],[]
    for key in keys:
        with Image.open(images[key]) as im:
            x=np.asarray(im.convert("RGB").resize(size,Image.Resampling.BILINEAR),dtype=np.float32)/255.
        with Image.open(masks[key]) as im:
            y=np.asarray(im.convert("L").resize(size,Image.Resampling.NEAREST),dtype=np.float32)
        xs.append(x);ys.append(y)
    return np.stack(xs),np.stack(ys)

ROOT="/content/drive/MyDrive/HITL"
X_train,Y_train=load_pairs(ROOT,"train")
test_dir=Path(ROOT)/"test"
if test_dir.exists():
    X_test,Y_test=load_pairs(ROOT,"test")
else:
    from sklearn.model_selection import train_test_split
    X_train,X_test,Y_train,Y_test=train_test_split(X_train,Y_train,test_size=0.2,random_state=42)
print(X_train.shape,Y_train.shape,X_test.shape,Y_test.shape)

## 2. Вывод загруженных изображений и масок

In [ ]:
fig,axs=plt.subplots(2,4,figsize=(12,6))
for i in range(min(4,len(X_train))):
    axs[0,i].imshow(X_train[i]);axs[0,i].axis("off")
    axs[1,i].imshow(Y_train[i],cmap="gray");axs[1,i].axis("off")
plt.tight_layout();plt.show()

## 3. Предобработка — допишите код

Для бинарной сегментации маски переводим в {0,1}. Если маски многоклассовые или цветные, этот шаг требует адаптации под разметку датасета. Выведите уникальные значения, чтобы проверить тип масок.

In [ ]:
print("Метки train:",np.unique(Y_train)[:30])
print("Метки test:",np.unique(Y_test)[:30])
# Для бинарной разметки порог выбирается относительно диапазона масок.
threshold=0.5 if max(float(Y_train.max()),float(Y_test.max()))<=1 else 127.5
Y_train=(Y_train>threshold).astype("float32")[...,None]
Y_test=(Y_test>threshold).astype("float32")[...,None]
print("После обработки:",X_train.shape,Y_train.shape)

## 4. Топология нейронной сети — U-Net для спутниковых снимков

Заполняем пропуски реализации Conv2D, pooling, upsampling, skip-connections, выходной слой sigmoid. Компилируем с бинарной кросс-энтропией и BinaryAccuracy.

In [ ]:
def conv_block(x,f):
    x=layers.Conv2D(f,3,padding="same",activation="relu")(x)
    return layers.Conv2D(f,3,padding="same",activation="relu")(x)
def make_unet(input_shape=(128,128,3),classes=1):
    inp=keras.Input(input_shape)
    c1=conv_block(inp,16);p1=layers.MaxPooling2D()(c1)
    c2=conv_block(p1,32);p2=layers.MaxPooling2D()(c2)
    bridge=conv_block(p2,64)
    u2=layers.UpSampling2D()(bridge);u2=layers.Concatenate()([u2,c2]);u2=conv_block(u2,32)
    u1=layers.UpSampling2D()(u2);u1=layers.Concatenate()([u1,c1]);u1=conv_block(u1,16)
    out=layers.Conv2D(classes,1,activation="sigmoid" if classes==1 else "softmax")(u1)
    return keras.Model(inp,out)

model=make_unet()
model.compile(optimizer=keras.optimizers.Adam(1e-3),loss="binary_crossentropy",metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])
model.summary()

## 5. Обучение — допишите код

In [ ]:
history=model.fit(X_train,Y_train,validation_split=0.2,epochs=10,batch_size=8,verbose=1)

## 6. Тестирование — допишите код

In [ ]:
metrics=model.evaluate(X_test,Y_test,return_dict=True,verbose=0)
print(metrics)

## 7. Результаты сегментации — допишите код

In [ ]:
pred=model.predict(X_test[:4],verbose=0)
fig,axs=plt.subplots(3,4,figsize=(12,9))
for i in range(min(4,len(pred))):
    axs[0,i].imshow(X_test[i]);axs[1,i].imshow(Y_test[i,:,:,0],cmap="gray");axs[2,i].imshow(pred[i,:,:,0],cmap="gray",vmin=0,vmax=1)
    for row in range(3):axs[row,i].axis("off")
plt.tight_layout();plt.show()
plt.plot(history.history["loss"],label="train");plt.plot(history.history["val_loss"],label="validation");plt.legend();plt.show()

### 7.1. IoU и Dice — анализ качества

In [ ]:
pred_bin=(model.predict(X_test,verbose=0)>0.5)
truth=Y_test>0.5
intersection=np.logical_and(pred_bin,truth).sum(axis=(1,2,3))
union=np.logical_or(pred_bin,truth).sum(axis=(1,2,3))
iou=np.mean((intersection+1e-7)/(union+1e-7))
dice=np.mean((2*intersection+1e-7)/(pred_bin.sum(axis=(1,2,3))+truth.sum(axis=(1,2,3))+1e-7))
print(f"IoU={iou:.4f}, Dice={dice:.4f}")

## Вывод

Опишите качество сегментации и расхождение train/validation; укажите IoU и Dice после выполнения. Результаты не придуманы. Обратите внимание: при масках с несколькими семантическими классами нужен отдельный многоклассовый вариант, а не бинаризация.